# Pipeline ablation — Plan × Audit, two SLMs, 300 held-out alerts

Ready to run on Colab (GPU T4). Nothing to edit: models are downloaded from the Hugging Face Hub,
the CIC-IoT-2023 artefacts are found on Drive as in the T_test2 notebook, and every intermediate
result is cached on Drive, so **if the session dies, reconnect and Run all again**: the run resumes.

**Question.** Does the full pipeline deliver fewer wrong or over-reaching statements per alert than
its parts, at what loss of content and at what cost?

**Design: 2 × 2 factorial on the same alerts.**

| | no audit | audit |
|---|---|---|
| **free generation** (SLM sees the record) | **A. Free** — delivered as written | **B. Free + check** — each sentence is verified; only SUPPORTED / CONSISTENT sentences are delivered |
| **planned generation** (SLM sees verified statements only) | **C. Plan** — delivered as written | **D. Plan + audit = full pipeline** — sentences that match no planned statement are removed; if none remains, the planned statements are delivered verbatim |

Only two texts are generated per alert and per SLM (free, planned); B and D re-use them, so the four
conditions differ only by the stages applied. The gate of prior work is not ablated (every alert is
explained in all four conditions).

**Measurement is independent of the verifier.** Conditions B and D use the compiler, so counting the
compiler's own refutations would make them look perfect by construction. The primary endpoints use
instruments that never call the compiler:
1. a **certifying parser** (exact schema names, strict syntax; the lower-bound parser of the theory
   paper), which recomputes the stated value or band from the record: *certified wrong facts* and
   *certified correct facts*;
2. an **over-reach lexicon** (intent, cause, certainty), written separately from the compiler's;
3. a **fabricated-feature lexicon** (network terms that are not in the 25-feature schema);
4. a **blind human annotation** of a stratified sample (section 9).

**Guards.** Compiler and helpers are hash-checked (identical to the T_test2 notebook); T_test is
replayed exactly; the alerts are the 300 T_test2 flows (never used to develop the pipeline); prompts,
planner, audit rule, lexicons and endpoints are hashed and frozen **before the first generation**.

**Run:** Runtime → GPU (T4), then Run all. ≈ 45–75 min the first time (generation), seconds afterwards.
Results: `/content/drive/MyDrive/project/ablation/`.

In [ ]:
# === 0. Setup =================================================================
import os, sys, re, json, time, hashlib, datetime, subprocess, warnings, gc
os.environ["TQDM_DISABLE"] = "1"; os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
DRYRUN = os.environ.get("ABL_DRYRUN") == "1"          # authors' plumbing test only (synthetic data, fake SLM)

class CFG:
    MODELS_ROOT = "/content/drive/MyDrive/models"
    TINY_DIR   = "/content/drive/MyDrive/models/TinyLlama_LoRA_r16_3ep_CICIoT2023"
    OUTPUT_DIR = "/content/drive/MyDrive/project/article_outputs"
    TTEST2_DIR = "/content/drive/MyDrive/project/ttest2"    # to check the alert set
    WORK       = "/content/drive/MyDrive/project/ablation"
    DRIVE_ROOT = "/content/drive/MyDrive"
    DATASET    = "CICIoT2023"
    SEED, SEED2 = 42, 2027          # T_dev/T_test flows ; T_test2 flows (the alerts used here)
    SLMS = {"TinyLlama-1.1B": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            "Qwen2.5-1.5B":   "Qwen/Qwen2.5-1.5B-Instruct"}
    MAX_NEW_TOKENS = 160            # greedy decoding
    BATCH      = 16
    CHUNK      = 32                 # alerts written to the Drive cache at a time
    N_LATENCY  = 20                 # single-alert calls timed per model and prompt
    K_FEATURES = 4                  # planned features per alert
    B_BOOT     = 10_000
    N_ANNOT    = 25                 # alerts per SLM in the blind annotation sheet (x 4 conditions)
    STRICT_REPRO = True

if DRYRUN:
    CFG.WORK = os.path.abspath(os.environ.get("ABL_WORK", "ablation_out"))
    CFG.OUTPUT_DIR = CFG.DRIVE_ROOT = CFG.TTEST2_DIR = CFG.WORK
    CFG.N_LATENCY = 3
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("Not on Colab or Drive not mounted:", e)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate", "joblib", "scikit-learn"], check=False)
os.makedirs(CFG.WORK, exist_ok=True)
CACHE = os.path.join(CFG.WORK, "cache"); os.makedirs(CACHE, exist_ok=True)
import numpy as np, pandas as pd
from scipy.stats import beta, binomtest
def tag(kind, msg): print(f"[{kind}] {msg}", flush=True)
tag("SETUP", f"results -> {CFG.WORK} | dryrun={DRYRUN}")

In [ ]:
# === Load the real artefacts (same code path as the T_test2 notebook) ==========
def load_live():
    import joblib, glob
    d = CFG.TINY_DIR
    if not os.path.exists(os.path.join(d, "label_encoder.joblib")):
        hits = [h for h in glob.glob(CFG.DRIVE_ROOT + "/**/label_encoder.joblib", recursive=True)
                if "TinyLlama" in h and CFG.DATASET in h]
        if not hits: raise FileNotFoundError("label_encoder.joblib not found: set CFG.TINY_DIR")
        d = CFG.TINY_DIR = os.path.dirname(hits[0]); tag("AUTO", f"TINY_DIR = {d}")
    le = joblib.load(os.path.join(d, "label_encoder.joblib")); classes = list(le.classes_)
    feats = list(joblib.load(os.path.join(d, "top_features.joblib")))
    scaler = joblib.load(os.path.join(d, "scaler.joblib"))
    Xraw = scaler.inverse_transform(np.load(os.path.join(d, "X_test_sc.npy")))
    return classes, feats, Xraw, np.load(os.path.join(d, "y_test.npy"))

def load_dryrun():
    rng = np.random.RandomState(0)
    feats = ["flow_duration", "Header_Length", "Protocol Type", "Duration", "Rate", "Srate", "Tot sum",
             "Min", "Max", "AVG", "Std", "Tot size", "IAT", "Magnitue", "Radius", "Covariance",
             "Variance", "Weight", "ack_count", "syn_count", "fin_count", "urg_count", "rst_count",
             "Drate", "Tot pkts"]
    classes = ["Benign", "DDoS", "DoS", "Mirai", "Recon", "Spoofing"]
    Xraw = np.abs(rng.lognormal(1.0, 1.2, size=(1800, len(feats)))); Xraw[rng.rand(*Xraw.shape) < .08] = 0
    return classes, feats, Xraw, rng.randint(0, len(classes), 1800)

classes, feats, Xraw, y = load_dryrun() if DRYRUN else load_live()
y_str = np.array([classes[int(i)] for i in y], dtype=object)
feat_q = {f: {"q33": float(np.percentile(Xraw[:, i], 33)), "q66": float(np.percentile(Xraw[:, i], 66)),
              "med": float(np.median(Xraw[:, i]))} for i, f in enumerate(feats)}
ATTACK = [c for c in classes if c.lower() != "benign"]
tag("DATA", f"{Xraw.shape[0]} flows x {len(feats)} features | classes={classes}")

## 1. Frozen verifier (verbatim, hash-checked) and V1

In [ ]:
FROZEN_COMPILER = r'''# === Claim compiler, router, and 4-verdict verifier =========================
import re
GRAMMAR=["VALUE","RANGE","COMPARISON","QUANTIFIER","ABSENCE","CLASS","BEHAVIOR","CAUSALITY"]
LOW={"low","short","small","brief","minimal","light"}; HIGH={"high","long","large","heavy","elevated","big"}
MED={"medium","moderate","average","typical"}; FEW={"few","only a few","little","fewer"}
# vague / underspecified magnitudes: no executable predicate -> must NOT be SUPPORTED
VAGUE={"unusual","atypical","abnormal","strange","irregular","notable","remarkable",
       "significant","considerable","relatively unusual","somewhat unusual","suspicious"}
MANY={"many","lots","numerous","sustained","high-volume","high volume","large number"}
BEHAV_NAMES={"reconnaissance","recon","exfiltration","beaconing","command-and-control","c2",
       "brute-force","bruteforce","scanning","port scan","portscan","probing","lateral movement",
       "denial of service","dos","ddos","volumetric","flooding","flood","spoofing","injection",
       "malware","botnet","mirai"}
BEHAV_CUES={"consistent with","indicative of","characteristic of","typical of","suggests",
       "pattern of","resembles","looks like","compatible with"}
BEHAV=BEHAV_NAMES|BEHAV_CUES
BENIGN={"benign","normal","legitimate","harmless","expected","safe"}
# keyword-independent causal / abusive-conclusion cues (explicit, implicit, inverted, conclusion)
CAUSAL_EXPLICIT={"caused","because","therefore","thus","hence","results in","leads to","due to"}
CAUSAL_IMPLICIT={"as a consequence","consequence of","resulted from","gives rise","stems from",
                 "attributable to","on account of","owing to"}
INTENT={"intent","intentional","deliberately","in order to","so as to","evade","evasion","attempts to"}
CONCLUSION={"is established","is confirmed","is certainly","certainly malicious","must be",
            "proves","demonstrates that","it follows that","can only mean"}
def real_values(j): return {feats[i]: float(Xraw[j][i]) for i in range(len(feats))}
def _has(words,c): return any(re.search(r"(?<![a-z])"+re.escape(w)+r"(?![a-z])",c) for w in words)
def _num(s):
    m=re.search(r"[-+]?\d+\.?\d*",s); return float(m.group()) if m else None
# --- feature synonym resolution: generated text rarely uses the exact schema name ---
SYN_TOKENS={"pkt":["packet","packets"],"len":["length","size"],"dur":["duration","time","lasted"],
            "fwd":["forward","outbound","upstream"],"bwd":["backward","inbound","downstream","response"],
            "iat":["inter-arrival","interarrival","arrival gap"],"tot":["total"],
            "byts":["bytes","volume"],"cnt":["count","number"],"avg":["average","mean"],
            "min":["minimum","smallest"],"max":["maximum","largest"],"std":["deviation","variance"],
            "rate":["rate","per second"],"flag":["flag","flags"],"hdr":["header"],"seg":["segment"]}
def _tok_syn(f):
    parts=f.lower().replace('_',' ').split()
    return [set([p])|set(SYN_TOKENS.get(p,[])) for p in parts]
_FEAT_TOKENS={f:_tok_syn(f) for f in feats}
def _feat(c):
    # score each schema feature by how many of its (synonym-expanded) tokens appear.
    # Requires a majority of tokens AND >=2 matches; ties -> None (ambiguous, conservative).
    best=[]; bestn=0
    for f in feats:
        toks=_FEAT_TOKENS[f]
        n=sum(1 for syns in toks if any(re.search(r"(?<![a-z])"+re.escape(w)+r"(?![a-z])",c) for w in syns))
        need=max(2, (len(toks)+1)//2) if len(toks)>1 else 1
        if n>=need:
            if n>bestn: best=[f]; bestn=n
            elif n==bestn: best.append(f)
    return best[0] if len(best)==1 else None
# a term that *looks like* a network feature but resolves to nothing in the schema
FEATUREISH={"entropy","handshake","tls","ssl","jitter","latency","throughput","payload",
            "checksum","ttl","window","retransmission","fragment","certificate","cipher"}
def is_causal_or_abusive(c):
    # structural: any causal/intent cue, OR a compound claim ending in an unwarranted conclusion
    if _has(CAUSAL_EXPLICIT,c) or _has(CAUSAL_IMPLICIT,c) or _has(INTENT,c) or _has(CONCLUSION,c):
        return True
    # conclusion pattern: "..., <claim>" with a malicious/benign verdict but no evidence operator
    if re.search(r"(malicious|attack|benign|normal)\b", c) and (";" in c or "," in c) and \
       not _feat(c): return True
    return False
def compile_claim(claim):
    c=claim.lower().strip()
    if is_causal_or_abusive(c):        return {"category":"CAUSALITY"}
    # CLASS first: "classified as Recon" / "predicts DDoS" is a decision restatement, not a
    # behavioral compatibility claim (previously mis-routed to CONSTRAINT on the keyword).
    _CLASS_NAMES={str(k).lower() for k in classes}
    _names_in = any(re.search(r"(?<![a-z])"+re.escape(k)+r"(?![a-z])", c) for k in _CLASS_NAMES)
    if (_has(BENIGN,c) or "classifier predicts" in c or "classified as" in c
        or "this is a" in c or "flow classified" in c or "labelled as" in c
        or "labeled as" in c): return {"category":"CLASS"}
    # decision restatements: "this flow is (likely) a DDoS attack", "it is a DoS", "indicates
    # a DoS attack" -- these assert the LABEL, not behavioural compatibility. Route to CLASS.
    if re.search(r"(is|are|be|was)\s+(indeed\s+|likely\s+|probably\s+|clearly\s+)?(an?\s+)?"
                 r"[a-z0-9\-]*\s*(attack|flow|traffic|activity)?", c) and (_names_in or _has(BEHAV_NAMES,c)) \
       and re.search(r"(this|the)\s+(flow|traffic|session|connection|sample)", c):
        return {"category":"CLASS"}
    if _names_in and not _has(BEHAV_CUES,c):            # bare class name, no compatibility cue
        return {"category":"CLASS"}
    if _has(BEHAV_CUES,c) and _has(BEHAV_NAMES,c):   # needs a compatibility cue AND a behavior name
        return {"category":"BEHAVIOR","behavior":next(w for w in BEHAV_NAMES if _has({w},c))}
    if _has(BEHAV_NAMES,c):            return {"category":"BEHAVIOR","behavior":next(w for w in BEHAV_NAMES if _has({w},c))}
    f=_feat(c)
    if f and _has({"no","zero","absent"},c):                 return {"category":"ABSENCE","feature":f}
    if f and _has(FEW|MANY,c):                               return {"category":"QUANTIFIER","feature":f,"dir":"few" if _has(FEW,c) else "many"}
    if f and _num(c) is not None and not _has(LOW|HIGH|MED,c):return {"category":"VALUE","feature":f,"operand":_num(c)}
    if f and _has(VAGUE,c) and not _has(LOW|HIGH|MED,c):
        return {"category":"VAGUE","feature":f}              # underspecified -> not executable
    if f and _has(LOW|HIGH|MED,c):
        d="low" if _has(LOW,c) else ("high" if _has(HIGH,c) else "medium")
        return {"category":"COMPARISON","feature":f,"dir":d}
    if f:                                                    return {"category":"RANGE","feature":f}
    return {"category":"CAUSALITY"}
def route(cat): return "EXEC" if cat in {"VALUE","RANGE","COMPARISON","QUANTIFIER","ABSENCE","CLASS"} \
                       else ("CONSTRAINT" if cat=="BEHAVIOR" else "SEMANTIC")   # VAGUE/CAUSALITY -> SEMANTIC -> abstain
def _band(f,v):
    q=feat_q[f]; return "low" if v<q["q33"] else ("high" if v>q["q66"] else "medium")
TOL=0.02
# Behaviour -> observable constraints. Each mapping states, in terms of the two flow
# observables we actually have (packet volume, duration), what the named behaviour would
# LOOK like. These are compatibility conditions, never proofs: full or partial satisfaction
# yields CONSISTENT, never SUPPORTED. Every name in BEHAV_NAMES must have an entry, or the
# branch silently returns UNRESOLVED and CONSISTENT can never fire.
BCON={
 # probing / enumeration: little data, short exchanges
 "reconnaissance":["few","short"], "recon":["few","short"],
 "scanning":["few","short"], "port scan":["few","short"], "portscan":["few","short"],
 "probing":["few","short"],
 # flooding / denial of service: high volume is the defining observable
 "denial of service":["many"], "dos":["many"], "ddos":["many"],
 "volumetric":["many"], "flooding":["many"], "flood":["many"],
 # bulk transfer out: high volume, sustained
 "exfiltration":["many","long"],
 # periodic low-rate callbacks
 "beaconing":["few"], "command-and-control":["few"], "c2":["few"],
 # repeated attempts: many small exchanges
 "brute-force":["many","short"], "bruteforce":["many","short"],
 # remaining families have no distinctive volume/duration signature in our feature set,
 # so we declare no constraint: the branch must abstain rather than invent one.
 "lateral movement":[], "spoofing":[], "injection":[], "malware":[], "botnet":[], "mirai":[],
}
_MISSING_BCON = {b for b in BEHAV_NAMES if b not in BCON}
assert not _MISSING_BCON, f"behaviour without a declared constraint set: {_MISSING_BCON}"
def _pf():
    for f in feats:
        if "pkt" in f.lower() or "packet" in f.lower(): return f
    return feats[0]
def _df():
    for f in feats:
        if "dur" in f.lower(): return f
    return feats[1] if len(feats)>1 else feats[0]
def verify(claim,j,pred=None):
    pred=pred if pred is not None else y_str[j]; rv=real_values(j)
    comp=compile_claim(claim); br=route(comp["category"]); cat=comp["category"]
    if br=="EXEC":
        if cat=="VALUE":
            f=comp["feature"]; return ("SUPPORTED" if abs(rv[f]-comp["operand"])<=TOL*max(abs(rv[f]),1e-6) else "CONTRADICTED"),br,cat
        if cat in ("COMPARISON","RANGE"):
            f=comp["feature"]; return ("SUPPORTED" if _band(f,rv[f])==comp.get("dir","medium") else "CONTRADICTED"),br,cat
        if cat=="QUANTIFIER":
            f=comp["feature"]; b=_band(f,rv[f]); w=comp["dir"]
            if (w=="few" and b=="low") or (w=="many" and b=="high"): return "SUPPORTED",br,cat
            if (w=="few" and b=="high") or (w=="many" and b=="low"): return "CONTRADICTED",br,cat
            return "UNRESOLVED",br,cat
        if cat=="ABSENCE":
            f=comp["feature"]; return ("SUPPORTED" if abs(rv[f])<1e-9 else "CONTRADICTED"),br,cat
        if cat=="CLASS":
            # Compare the class the claim NAMES against the decision, instead of assuming
            # benign wording: "this flow is a DDoS attack" restates an attack decision
            # correctly, while benign wording on an attack decision contradicts it.
            cl=claim.lower()
            named=[k for k in classes
                   if re.search(r"(?<![a-z])"+re.escape(str(k).lower())+r"(?![a-z])", cl)]
            if named:
                return ("SUPPORTED" if any(str(k).lower()==str(pred).lower() for k in named)
                        else "CONTRADICTED"), br, cat
            if _has(BENIGN,cl):                 # benign wording, no explicit class name
                return ("CONTRADICTED" if pred in ATTACK else "SUPPORTED"),br,cat
            return "UNRESOLVED",br,cat          # a class statement naming nothing checkable
    if br=="CONSTRAINT":
        cons=BCON.get(comp.get("behavior",""),[])
        if not cons: return "UNRESOLVED",br,cat      # no declared observable signature -> abstain
        pf,df=_pf(),_df(); sat=0
        for k in cons:
            if k=="few" and _band(pf,rv[pf])=="low": sat+=1
            elif k=="many" and _band(pf,rv[pf])=="high": sat+=1
            elif k=="medium" and _band(pf,rv[pf])=="medium": sat+=1
            elif k=="short" and _band(df,rv[df])=="low": sat+=1
            elif k=="long" and _band(df,rv[df])=="high": sat+=1
        # A behaviour is never *entailed* by one flow, so full satisfaction is still only
        # CONSISTENT. Partial satisfaction is also CONSISTENT (evidence compatible, not
        # probative); zero satisfaction contradicts the stated behaviour.
        score=sat/len(cons)
        if score==0.0:  return "CONTRADICTED",br,cat
        if score>=0.5:  return "CONSISTENT",br,cat
        return "UNRESOLVED",br,cat
    return "UNRESOLVED",br,cat            # SEMANTIC/CAUSALITY -> abstain
for ex in ["The traffic is benign and normal.",
           "These statistics indicate reconnaissance.",
           "The attack occurred as a consequence of the limited packet exchange.",
           "Given the short flow and low packet count, malicious intent is established."]:
    print(f"  {ex[:52]:54s} -> {verify(ex,int(0))}")
tag("VERIFIER","4 verdicts {SUPPORTED,CONTRADICTED,CONSISTENT,UNRESOLVED}; causality keyword-independent")'''
assert hashlib.sha256(FROZEN_COMPILER.encode()).hexdigest() == "83ccacc6ca15b6fc50dee17d07154e44f7bd81807cb25d827e1133ef9aa2a7c8", "compiler differs: STOP"
exec(FROZEN_COMPILER)
tag("FROZEN", "compiler/router/branches (sha256 ok, same as T_test2)")

In [ ]:
FROZEN_HELPERS = r'''# Discourse / meta sentences are not claims about the flow: they state what one *would*
# do, ask a question, or introduce the answer. Auditing them produces false flags.
META_CUES={"we need to consider","to make a decision","in terms of","let us","let's",
           "in order to determine","the following","as follows","in summary","to summarize",
           "based on the above","here is","i will","we can use","we should","it is important",
           "note that","for example","in this case we","the goal is","first,","second,",
           "step 1","step 2","explanation:","answer:","justification:"}
def is_meta(s):
    c=s.lower().strip()
    if c.endswith("?"): return True                      # a question asserts nothing
    if _has(META_CUES,c) or any(k in c for k in META_CUES): return True
    if re.match(r"^\s*(to|in order to|how|why|what|when)\b", c): return True
    return False

def split_sentences(text):
    # Generator output is not reliably punctuated: a trailing sentence without a final
    # period would otherwise be merged with the next one, so the audit would analyse a
    # composite string and silently miss a claim. Split on terminal punctuation AND on
    # newlines / bullet markers, then re-split overlong fragments on " The <feature>".
    t=re.sub(r"\s*[\r\n]+\s*", " . ", str(text).strip())
    t=re.sub(r"\s+[-*\u2022]\s+", " . ", t)
    parts=re.split(r"(?<=[.!?])\s+", t)
    out=[]
    for p in parts:
        p=p.strip()
        if not p: continue
        # a fragment holding several "The <something> is ..." assertions -> split them
        if len(p.split())>25 and len(re.findall(r"(?i)\bthe\s+[a-z_ ]{2,30}\s+is\b", p))>1:
            sub=re.split(r"(?i)(?=\bthe\s+[a-z_ ]{2,30}\s+is\b)", p)
            out+= [x.strip() for x in sub if len(x.split())>=3]
        else:
            out.append(p)
    return [x.strip(" .") for x in out if len(x.split())>=3]

'''
assert hashlib.sha256(FROZEN_HELPERS.encode()).hexdigest() == "4b24155ec040964b884dad9bd5a39e3da7c88e68d27073803b36ac78e9643a50"
exec(FROZEN_HELPERS)
tag("FROZEN", "is_meta / split_sentences (sha256 ok)")

In [ ]:
# V1 = V0 with the single structural rule of the paper: an operator-free feature
# mention (compiled RANGE) is sent to abstention.
def V1(text, j, pred=None):
    v, br, cat = verify(text, j, pred)
    if cat == "RANGE": return "UNRESOLVED", "SEMANTIC", "OPFREE"
    return v, br, cat
OK_VERDICTS = {"SUPPORTED", "CONSISTENT"}
GROUP = {"VALUE": "VAL", "RANGE": "CMP", "COMPARISON": "CMP", "QUANTIFIER": "QNT",
         "ABSENCE": "ABN", "CLASS": "CLS", "BEHAVIOR": "BEH"}

## 2. Guards: T_test replayed exactly, alerts = the 300 T_test2 flows

In [ ]:
FROZEN_BANK = r'''# === Template banks: T_dev (seen) and T_test (UNSEEN, disjoint) =============
# Each entry: (category, builder(j) -> (text, gold_verdict))
def _f0(): return feats[0]
def _f1(): return feats[1] if len(feats)>1 else feats[0]
def _nm(f): return f.replace('_',' ')
INV={"low":"high","high":"low","medium":"medium"}

T_dev=[
 ("COMPARISON", lambda j: (f"The {_nm(_f0())} is {_band(_f0(),real_values(j)[_f0()])} for this dataset.","SUPPORTED")),
 ("COMPARISON", lambda j: (f"The {_nm(_f0())} is {INV[_band(_f0(),real_values(j)[_f0()])]} for this dataset.",
                           "CONTRADICTED" if _band(_f0(),real_values(j)[_f0()])!="medium" else "SUPPORTED")),
 ("VALUE",      lambda j: (f"The {_nm(_f0())} is {round(real_values(j)[_f0()],3)}.","SUPPORTED")),
 ("QUANTIFIER", lambda j: (f"The flow shows few {_nm(_f1())}.",
                           {"low":"SUPPORTED","high":"CONTRADICTED","medium":"UNRESOLVED"}[_band(_f1(),real_values(j)[_f1()])])),
 ("CLASS",      lambda j: (f"The classifier predicts {y_str[j]}.","SUPPORTED")),
 ("BEHAVIOR",   lambda j: ("These statistics indicate reconnaissance.", None)),   # gold = CONSISTENT/CONTRA per flow
 ("CAUSALITY",  lambda j: (f"The low {_nm(_f1())} caused the observed attack.","UNRESOLVED")),
]

# --- UNSEEN paraphrases: different syntax and lexicon, same semantics -------------
T_test=[
 # COMPARISON: hedged, inverted, comparative-clause phrasings
 ("COMPARISON", lambda j: (f"Compared with the rest of the corpus, the {_nm(_f0())} sits in the "
                           f"{_band(_f0(),real_values(j)[_f0()])} band.","SUPPORTED")),
 ("COMPARISON", lambda j: (f"Relative to the dataset, this sample's {_nm(_f0())} registers as "
                           f"{INV[_band(_f0(),real_values(j)[_f0()])]}.",
                           "CONTRADICTED" if _band(_f0(),real_values(j)[_f0()])!="medium" else "SUPPORTED")),
 # VALUE: passive / measurement phrasing
 ("VALUE",      lambda j: (f"A {_nm(_f0())} of {round(real_values(j)[_f0()],3)} was recorded for this flow.","SUPPORTED")),
 ("VALUE",      lambda j: (f"This flow's {_nm(_f0())} was measured at {round(real_values(j)[_f0()]*3.1+7.0,3)}.","CONTRADICTED")),
 # QUANTIFIER: unseen quantifier lexicon
 ("QUANTIFIER", lambda j: (f"Only a handful of {_nm(_f1())} appear in this exchange.",
                           {"low":"SUPPORTED","high":"CONTRADICTED","medium":"UNRESOLVED"}[_band(_f1(),real_values(j)[_f1()])])),
 ("QUANTIFIER", lambda j: (f"This exchange carries numerous {_nm(_f1())}.",
                           {"high":"SUPPORTED","low":"CONTRADICTED","medium":"UNRESOLVED"}[_band(_f1(),real_values(j)[_f1()])])),
 # CLASS: unseen restatement forms
 ("CLASS",      lambda j: (f"This session was labelled as {y_str[j]} by the collective model.","SUPPORTED")),
 ("CLASS",      lambda j: ("The session appears entirely legitimate and unremarkable.",
                           "CONTRADICTED" if y_str[j] in ATTACK else "SUPPORTED")),
 # BEHAVIOR: unseen compatibility cues
 ("BEHAVIOR",   lambda j: ("The observed pattern is characteristic of scanning.", None)),
 # CAUSALITY: unseen implicit / inverted constructions (no 'caused')
 ("CAUSALITY",  lambda j: (f"The sparse {_nm(_f1())} is what gave rise to this incident.","UNRESOLVED")),
 ("CAUSALITY",  lambda j: ("It follows from these measurements that the operator acted with hostile intent.","UNRESOLVED")),
 ("CAUSALITY",  lambda j: ("This session must therefore form part of a coordinated campaign.","UNRESOLVED")),
]
print(f"T_dev: {len(T_dev)} templates | T_test: {len(T_test)} templates")
# disjointness is by construction (different surface forms); assert no literal overlap
_d={t(0)[0] for _,t in T_dev}; _t={t(0)[0] for _,t in T_test}
assert not (_d & _t), "template leakage between dev and test"
print("T_dev intersect T_test = empty  ->  no template leakage")'''
assert hashlib.sha256(FROZEN_BANK.encode()).hexdigest() == "db273bc386d473ea76152ad17c5de4c1b19c660be6f77c9f50a1f55a126874d8"
exec(FROZEN_BANK)

In [ ]:
rng = np.random.RandomState(CFG.SEED)
flows = [int(x) for x in rng.choice(Xraw.shape[0], min(300, Xraw.shape[0]), replace=False)]
def bcon_gold(name, j):
    rv = real_values(j); cons = BCON.get(name, [])
    if not cons: return "UNRESOLVED"
    pf, df_ = _pf(), _df(); sat = 0
    for k in cons:
        if   k == "few"    and _band(pf, rv[pf]) == "low":     sat += 1
        elif k == "many"   and _band(pf, rv[pf]) == "high":    sat += 1
        elif k == "medium" and _band(pf, rv[pf]) == "medium":  sat += 1
        elif k == "short"  and _band(df_, rv[df_]) == "low":   sat += 1
        elif k == "long"   and _band(df_, rv[df_]) == "high":  sat += 1
    s = sat / len(cons)
    return "CONTRADICTED" if s == 0 else ("CONSISTENT" if s >= .5 else "UNRESOLVED")
n = ok = fs = 0
for cat, builder in T_test:
    for j in flows:
        text, gold = builder(j)
        if gold is None: gold = bcon_gold(next(w for w in BCON if w in text.lower()), j)
        v, br, c = verify(text, j); n += 1; ok += (v == gold); fs += (v == "SUPPORTED" and gold != "SUPPORTED")
REPRO = dict(n=n, acc=round(ok / n, 3), errors=n - ok, false_supported=fs)
tag("REPRO", f"T_test: {REPRO}  (paper: n=3600, acc=0.864, errors=489, false_supported=218)")
if not DRYRUN and (n, n - ok, fs) != (3600, 489, 218):
    msg = "T_test is not reproduced: the artefacts differ from the paper's run."
    if CFG.STRICT_REPRO: raise RuntimeError(msg)
    print("WARNING:", msg)

# Alerts: the 300 T_test2 flows (seed 2027, disjoint from every flow used in development)
rng2 = np.random.default_rng(CFG.SEED2)
pool = sorted(set(range(Xraw.shape[0])) - set(flows))
ALERTS = sorted(int(x) for x in rng2.choice(pool, 300, replace=False))
ck = os.path.join(CFG.TTEST2_DIR, "ttest2_claims.csv")
if os.path.exists(ck):
    t2 = pd.read_csv(ck, usecols=["set", "rec"]); t2 = set(t2[t2.set == "ttest2"].rec.astype(int))
    assert t2 == set(ALERTS), "the alert set differs from the T_test2 flows: STOP"
    tag("ALERTS", "300 alerts = exactly the T_test2 flows (checked against ttest2_claims.csv)")
else:
    tag("ALERTS", "300 alerts rebuilt with the T_test2 seed (ttest2_claims.csv not found for the cross-check)")
assert not set(ALERTS) & set(flows)

## 3. Protocol: planner, prompts, audit rule, instruments — frozen before generation

* **Decision ŷ** = the label used as the detector decision in all earlier notebooks (`y_test`); the
  explanation stage is evaluated given ŷ, the detector's accuracy is out of scope.
* **Plan.** For each alert: the decision; for up to 4 features whose band is low or high (schema order
  = mutual-information rank) a VALUE statement and a COMPARISON statement; one hedged behaviour
  statement when the decision maps to a declared behaviour.
* **Verify (V1).** A planned statement enters C* only if V1 returns SUPPORTED (or CONSISTENT for the
  behaviour statement). Statements the compiler cannot resolve are dropped, never forced through.
* **Same information for both prompts:** the free prompt receives all 25 values *and* their bands, so
  the comparison isolates planning and audit, not access to information.

In [ ]:
def disp(f): return f.replace("_", " ")
def fmt(x): return np.format_float_positional(float(x), precision=4, unique=False, fractional=False, trim="-")
BEHAVIOUR_OF = [("ddos", "denial of service"), ("dos", "denial of service"), ("recon", "scanning"),
                ("brute", "brute-force"), ("scan", "scanning")]
def behaviour_for(label):
    l = str(label).lower()
    return next((b for k, b in BEHAVIOUR_OF if k in l), None)

SYSTEM = "You are an assistant that explains network intrusion alerts to security analysts."
PROMPT_FREE = ("A network intrusion detector classified a flow as {yhat}.\n"
               "Flow record (value, and band relative to other flows):\n{record}\n\n"
               "Explain to a security analyst, in 3 to 5 sentences, why this flow was classified as {yhat}.")
PROMPT_PLAN = ("A network intrusion detector classified a flow as {yhat}.\n"
               "The following statements have been verified against the flow record:\n{facts}\n\n"
               "Write a short explanation (3 to 5 sentences) for a security analyst using only these "
               "statements. Do not add any other number, feature, cause or intention. Keep the word "
               "'compatible' where it appears.")

def record_text(j):
    rv = real_values(j)
    return "\n".join(f"- {disp(f)}: {fmt(rv[f])} ({_band(f, rv[f])})" for f in feats)

def plan(j):
    # candidate statements from the record and the decision (Plan), then V1 (Verify)
    rv, yhat = real_values(j), y_str[j]; cand = [("CLASS", None, f"The detector classified this flow as {yhat}.")]
    picked = [f for f in feats if _band(f, rv[f]) != "medium"][:CFG.K_FEATURES]
    for f in picked:
        cand.append(("VALUE", f, f"The {disp(f)} is {fmt(rv[f])}."))
        cand.append(("COMPARISON", f, f"The {disp(f)} is {_band(f, rv[f])} compared with other flows."))
    b = behaviour_for(yhat)
    if b: cand.append(("BEHAVIOR", None, f"Its packet volume and duration are compatible with {b}."))
    kept, dropped = [], []
    for typ, f, s in cand:
        v, br, cat = V1(s, j, yhat); comp = compile_claim(s)
        good = (v == "SUPPORTED") or (typ == "BEHAVIOR" and v == "CONSISTENT")
        (kept if good else dropped).append(dict(text=s, typ=typ, feature=comp.get("feature"),
                                                dir=comp.get("dir"), behavior=comp.get("behavior"),
                                                cat=cat, verdict=v))
    return kept, dropped

def prompt_free(j): return PROMPT_FREE.format(yhat=y_str[j], record=record_text(j))
def prompt_plan(j, kept):
    return PROMPT_PLAN.format(yhat=y_str[j], facts="\n".join(f"- {p['text']}" for p in kept))

In [ ]:
# --- delivery rules of the four conditions ------------------------------------
def audit_match(s, j, cstar):
    # Paper, Sec. III-B: a sentence passes the audit iff its typed content matches a statement of C*
    # (same feature, type and direction) and the verifier accepts it.
    v, br, cat = V1(s, j, y_str[j])
    if v not in OK_VERDICTS: return False
    comp = compile_claim(s)
    for p in cstar:
        if GROUP.get(cat) != GROUP.get(p["cat"]): continue
        if cat == "CLASS": return True
        if cat == "BEHAVIOR" and comp.get("behavior") == p["behavior"]: return True
        if cat in ("VALUE", "COMPARISON", "QUANTIFIER", "ABSENCE") and comp.get("feature") == p["feature"]:
            if cat == "VALUE" or comp.get("dir") == p["dir"]: return True
    return False

def deliver(cond, j, free_text, plan_text, cstar):
    # returns (delivered sentences, flags); timing measured outside
    if cond == "A": return split_sentences(free_text), {}
    if cond == "C": return split_sentences(plan_text), {}
    if cond == "B":
        keep = [s for s in split_sentences(free_text)
                if not is_meta(s) and V1(s, j, y_str[j])[0] in OK_VERDICTS]
        return keep, {"empty": len(keep) == 0}
    if cond == "D":
        sents = split_sentences(plan_text)
        keep = [s for s in sents if not is_meta(s) and audit_match(s, j, cstar)]
        flagged = len([s for s in sents if not is_meta(s)]) - len(keep)
        if not keep:
            return [p["text"].rstrip(".") for p in cstar], {"fallback": True, "flagged": flagged}
        return keep, {"fallback": False, "flagged": flagged}
    raise ValueError(cond)
CONDS = {"A": "Free", "B": "Free + check", "C": "Plan", "D": "Plan + audit (full)"}

In [ ]:
# --- independent instruments (never call compile_claim / verify) ---------------
NEG = r"\b(not|no|never|neither|nor|without)\b"
DEICTIC = r"\b(this|that|the|each)\s+(feature|value|metric|field)\b"
AMBIG = r"\b(medium|moderate|average|typical|normal|usual|mid)\b"
HIGHW, LOWW = {"high", "large"}, {"low", "small"}
COMMON_WORDS = {"rate", "number", "weight", "duration", "radius", "variance", "covariance", "min", "max"}
NAMES = sorted({disp(f): f for f in feats}.items(), key=lambda kv: -len(kv[0]))

def certify(sentence):
    # Certifying parser of the theory paper (appendix C), used here stand-alone: returns
    # (feature, 'value', v, decimals) | (feature, 'band', 'high'|'low') | None
    s = re.sub(r"[*_`#>]", " ", sentence)
    if ";" in s or "%" in s: return None
    low = s.lower()
    if re.search(NEG, low) or re.search(DEICTIC, low) or re.search(AMBIG, low): return None
    hits, masked = [], s
    for name, f in NAMES:
        flags = 0 if name.lower() in COMMON_WORDS else re.I
        pat = r"(?<![A-Za-z0-9])" + re.escape(name) + r"(?![A-Za-z0-9])"
        if re.search(pat, masked, flags):
            hits.append(f); masked = re.sub(pat, " ", masked, flags=flags)
    if len(hits) != 1: return None
    f = hits[0]
    nums = re.findall(r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?", masked)
    words = set(re.findall(r"[a-z]+", masked.lower()))
    hb, lb = bool(words & HIGHW), bool(words & LOWW)
    if hb != lb and len(nums) <= 1: return (f, "band", "high" if hb else "low")
    if not hb and not lb and len(nums) == 1:
        d = len(nums[0].split(".")[1]) if "." in nums[0] else 0
        return (f, "value", float(nums[0]), d)
    return None

def certified_truth(c, j):
    x = real_values(j)[c[0]]
    if c[1] == "band": return _band(c[0], x) == c[2]
    v, d = c[2], c[3]
    return abs(v - x) <= max(TOL * abs(x), 0.5 * 10 ** (-d))         # precision-aware tolerance

OVERREACH = [r"\bintent", r"\bintend", r"\bdeliberate", r"\battacker", r"\badversar", r"\bmalicious actor",
             r"\bcampaign", r"\bexfiltrat", r"\bcompromis", r"\binfect", r"\bsteal", r"\bin order to\b",
             r"\baims? to\b", r"\btrying to\b", r"\bcaus(?:e|ed|es|ing)\b", r"\bbecause\b", r"\bdue to\b",
             r"\bresult(?:ed|s)? (?:in|from)\b", r"\b(?:led|leads?) to\b", r"\bclearly\b", r"\bdefinitely\b",
             r"\bcertainly\b", r"\bproves?\b", r"\bundoubtedly\b", r"\bconfirms? that\b"]
_schema_low = " ".join(disp(f).lower() for f in feats)
FABRICATED = [t for t in ["entropy", "payload", "tls", "ssl", "certificate", "jitter", "latency", "throughput",
                          "checksum", "window size", "retransmission", "source ip", "destination ip",
                          "ip address", "port number", "user agent", "url", "packet loss", "bandwidth",
                          "geolocation", "signature"] if t not in _schema_low]
def overreach(s): return any(re.search(p, s.lower()) for p in OVERREACH)
def fabricated(s): return any(re.search(r"\b" + re.escape(t) + r"\b", s.lower()) for t in FABRICATED)

def measure(sents, j):
    cw = cr = orc = fab = 0; right_feats = set(); v1ok = v1ref = 0
    for s in sents:
        c = certify(s)
        if c is not None:
            if certified_truth(c, j): cr += 1; right_feats.add(c[0])
            else: cw += 1
        orc += overreach(s); fab += fabricated(s)
        v, br, _ = V1(s, j, y_str[j]); v1ok += v in OK_VERDICTS; v1ref += (br == "EXEC" and v == "CONTRADICTED")
    return dict(sentences=len(sents), words=sum(len(s.split()) for s in sents), cert_wrong=cw, cert_right=cr,
                cert_right_features=len(right_feats), overreach=orc, fabricated=fab,
                v1_accepted=v1ok, v1_refuted=v1ref)

for s in ["The Tot sum of this flow is 512.", "The Rate is high compared with other flows.",
          "The rate is very high, which caused the alert.", "Its payload entropy is unusual."]:
    print(f"  {s:56s} certify={certify(s)} overreach={overreach(s)} fabricated={fabricated(s)}")

In [ ]:
PROTOCOL = dict(system=SYSTEM, prompt_free=PROMPT_FREE, prompt_plan=PROMPT_PLAN, k_features=CFG.K_FEATURES,
                behaviour_of=BEHAVIOUR_OF, slms=CFG.SLMS, max_new_tokens=CFG.MAX_NEW_TOKENS, decoding="greedy",
                batch=CFG.BATCH, alerts=ALERTS, overreach=OVERREACH, fabricated=FABRICATED,
                certifier=[NEG, DEICTIC, AMBIG, sorted(HIGHW), sorted(LOWW), sorted(COMMON_WORDS)],
                audit_rule="same group; CLASS; same behaviour; same feature and (VALUE or same dir); verdict in OK",
                conditions=CONDS, compiler_sha=hashlib.sha256(FROZEN_COMPILER.encode()).hexdigest(),
                primary_endpoints=["alerts with >=1 certified wrong fact", "alerts with >=1 over-reach sentence",
                                   "alerts with >=1 fabricated-feature sentence", "alerts with any of the three",
                                   "certified correct features per alert"])
FREEZE = os.path.join(CFG.WORK, "ablation_freeze.json")
SHA = hashlib.sha256(json.dumps(PROTOCOL, sort_keys=True, default=str).encode()).hexdigest()
if os.path.exists(FREEZE):
    prev = json.load(open(FREEZE))
    if prev["sha256"] != SHA:
        raise RuntimeError(f"Protocol changed after it was frozen on {prev['frozen_utc']}: STOP.")
    tag("FREEZE", f"protocol frozen on {prev['frozen_utc']} (sha {SHA[:16]})")
else:
    json.dump(dict(sha256=SHA, frozen_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                   protocol=PROTOCOL), open(FREEZE, "w"), indent=1, default=str)
    tag("FREEZE", f"protocol FROZEN now (sha {SHA[:16]}) -> {FREEZE}")

## 4. Plan + Verify on the 300 alerts (CPU)

In [ ]:
PLANS = {}; t0 = time.perf_counter()
for j in ALERTS: PLANS[j] = plan(j)
PLAN_MS = (time.perf_counter() - t0) * 1e3 / len(ALERTS)
kept = [len(PLANS[j][0]) for j in ALERTS]; dropped = sum(len(PLANS[j][1]) for j in ALERTS)
tag("PLAN", f"C*: {np.mean(kept):.2f} statements per alert (min {min(kept)}, max {max(kept)}); "
            f"{dropped} candidates dropped by V1; {PLAN_MS:.2f} ms per alert")
drop_types = pd.Series([d["typ"] for j in ALERTS for d in PLANS[j][1]]).value_counts().to_dict()
print("dropped candidates by type:", drop_types)
j0 = ALERTS[0]; print("\nExample, alert", j0, "-", y_str[j0]); print(prompt_plan(j0, PLANS[j0][0]))

## 5. Generation (GPU, resumable)

Two prompts × 300 alerts × 2 SLMs, batched greedy decoding, cached on Drive every 32 alerts.
Latency is measured separately on single-alert calls (the deployment setting), with peak GPU memory.

In [ ]:
def cache_path(slm, kind): return os.path.join(CACHE, f"gen__{slm}__{kind}.csv")
def cached(slm, kind):
    p = cache_path(slm, kind)
    if not os.path.exists(p): return {}
    c = pd.read_csv(p, keep_default_na=False).drop_duplicates("rec", keep="last")
    return dict(zip(c.rec.astype(int), c.text.astype(str)))
def append(slm, kind, recs, texts):
    p = cache_path(slm, kind)
    pd.DataFrame({"rec": recs, "text": texts}).to_csv(p, mode="a", header=not os.path.exists(p), index=False)
COST_PATH = os.path.join(CACHE, "cost.json")
COST = json.load(open(COST_PATH)) if os.path.exists(COST_PATH) else {}
def save_cost(): json.dump(COST, open(COST_PATH, "w"), indent=1)

class FakeSLM:   # DRYRUN only: copies the facts, sometimes adds an unplanned or over-reaching sentence
    def __init__(self, seed): self.g = np.random.default_rng(seed)
    def __call__(self, prompts):
        out = []
        for p in prompts:
            lines = [l[2:] for l in p.splitlines() if l.startswith("- ")]
            if "verified" in p:
                txt = " ".join(lines)
            else:
                pick = [l.split(":")[0] + " is " + l.split(":")[1].strip().split(" ")[0] + "." for l in lines[:3]]
                txt = " ".join(pick)
            r = self.g.random()
            if r < .3: txt += " The attacker clearly intended to overload the server."
            elif r < .5: txt += f" The {disp(feats[0])} is 999.5."
            elif r < .6: txt += " Its payload entropy is unusual."
            out.append(txt)
        return out

def load_model(hub_id):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(hub_id); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(hub_id, torch_dtype=torch.float16).to("cuda").eval()
    gc_ = model.generation_config; gc_.temperature = gc_.top_p = gc_.top_k = None; gc_.do_sample = False
    def gen(prompts):
        texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}],
                                         tokenize=False, add_generation_prompt=True) for p in prompts]
        enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=CFG.MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]
    return model, tok, gen

if not DRYRUN:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU: Runtime > Change runtime type > T4 GPU, then Run all.")
    tag("GPU", torch.cuda.get_device_name(0))

PROMPTS = {"free": {j: prompt_free(j) for j in ALERTS}, "plan": {j: prompt_plan(j, PLANS[j][0]) for j in ALERTS}}
for slm, hub in CFG.SLMS.items():
    need = any(len(cached(slm, k)) < len(ALERTS) for k in PROMPTS) or f"{slm}|free|ms" not in COST
    if not need: tag("CACHE", f"{slm}: complete"); continue
    if DRYRUN: gen = FakeSLM(len(slm)); model = None
    else:
        tag("MODEL", f"loading {hub}"); model, tok, gen = load_model(hub)
    for kind, P in PROMPTS.items():
        done = cached(slm, kind); todo = [j for j in ALERTS if j not in done]; t0 = time.time()
        for a in range(0, len(todo), CFG.CHUNK):
            part = todo[a:a + CFG.CHUNK]; texts = []
            for b in range(0, len(part), CFG.BATCH):
                texts += gen([P[j] for j in part[b:b + CFG.BATCH]])
            append(slm, kind, part, texts); k = a + len(part); el = time.time() - t0
            print(f"  {slm} | {kind}: {len(ALERTS) - len(todo) + k}/{len(ALERTS)}  ({el / 60:.1f} min, "
                  f"~{el / k * (len(todo) - k) / 60:.1f} min left)", flush=True)
    if f"{slm}|free|ms" not in COST:          # single-alert latency and peak memory
        if not DRYRUN: torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        for kind, P in PROMPTS.items():
            ts = []
            for j in ALERTS[:CFG.N_LATENCY]:
                t0 = time.perf_counter(); gen([P[j]])
                if not DRYRUN: torch.cuda.synchronize()
                ts.append((time.perf_counter() - t0) * 1e3)
            COST[f"{slm}|{kind}|ms"] = float(np.median(ts))
        COST[f"{slm}|peak_gpu_MB"] = (float(torch.cuda.max_memory_allocated() / 2**20) if not DRYRUN else 0.0)
        save_cost()
    if model is not None:
        del model, gen; gc.collect(); torch.cuda.empty_cache()
print({k: round(v, 1) for k, v in COST.items()})

## 6. Delivery under the four conditions, and measurement (CPU)

In [ ]:
ROWS, EXAMPLES = [], []
for slm in CFG.SLMS:
    FREE, PLANT = cached(slm, "free"), cached(slm, "plan")
    assert len(FREE) == len(PLANT) == len(ALERTS), f"{slm}: generation incomplete, re-run section 5"
    for j in ALERTS:
        cstar = PLANS[j][0]
        for cond in CONDS:
            t0 = time.perf_counter(); sents, flags = deliver(cond, j, FREE[j], PLANT[j], cstar)
            ms = (time.perf_counter() - t0) * 1e3
            m = measure(sents, j)
            ROWS.append(dict(slm=slm, cond=cond, rec=j, yhat=y_str[j], check_ms=ms,
                             is_empty=bool(flags.get("empty", False)), is_fallback=bool(flags.get("fallback", False)),
                             flagged=int(flags.get("flagged", 0)), text=" ".join(s + "." for s in sents), **m))
D = pd.DataFrame(ROWS)
D["any_wrong"] = D.cert_wrong > 0; D["any_overreach"] = D.overreach > 0; D["any_fabricated"] = D.fabricated > 0
D["any_problem"] = D.any_wrong | D.any_overreach | D.any_fabricated
D.to_csv(os.path.join(CFG.WORK, "ablation_alerts.csv"), index=False)
tag("SAVE", f"{len(D)} delivered explanations -> ablation_alerts.csv")
for slm in CFG.SLMS:
    j = ALERTS[1]; print(f"\n=== {slm}, alert {j} ({y_str[j]}) ===")
    for cond in CONDS:
        r = D[(D.slm == slm) & (D.cond == cond) & (D.rec == j)].iloc[0]
        print(f"[{cond}] {r.text[:400]}")

## 7. Results: alert-level endpoints, paired comparisons, cost

In [ ]:
def cp(k, n, a=.05):
    lo = 0.0 if k == 0 else float(beta.ppf(a / 2, k, n - k + 1)); hi = 1.0 if k == n else float(beta.ppf(1 - a / 2, k + 1, n - k))
    return round(lo, 3), round(hi, 3)
def boot_diff(a, b, B=CFG.B_BOOT, seed=CFG.SEED2):
    g = np.random.default_rng(seed); d = a.astype(float) - b.astype(float); n = len(d)
    est = np.array([d[g.integers(0, n, n)].mean() for _ in range(B)])
    return round(float(d.mean()), 3), [round(float(x), 3) for x in np.percentile(est, [2.5, 97.5])]

SUMMARY = []
for slm in CFG.SLMS:
    for cond in CONDS:
        s = D[(D.slm == slm) & (D.cond == cond)].sort_values("rec"); n = len(s)
        row = dict(slm=slm, cond=cond, name=CONDS[cond], n=n)
        for k in ["any_wrong", "any_overreach", "any_fabricated", "any_problem"]:
            row[k] = int(s[k].sum()); row[k + "_rate"] = round(s[k].mean(), 3); row[k + "_ci"] = cp(int(s[k].sum()), n)
        row.update(cert_right_features=round(s.cert_right_features.mean(), 2), v1_accepted=round(s.v1_accepted.mean(), 2),
                   sentences=round(s.sentences.mean(), 2), words=round(s.words.mean(), 1),
                   empty_rate=round(s.is_empty.mean(), 3), fallback_rate=round(s.is_fallback.mean(), 3),
                   flagged_per_alert=round(s.flagged.mean(), 2), check_ms=round(float(s.check_ms.median()), 2))
        gen_kind = "free" if cond in "AB" else "plan"
        row["gen_ms"] = round(COST.get(f"{slm}|{gen_kind}|ms", float("nan")), 0)
        row["plan_ms"] = round(PLAN_MS, 2) if cond in "CD" else 0.0
        row["peak_gpu_MB"] = round(COST.get(f"{slm}|peak_gpu_MB", float("nan")), 0)
        SUMMARY.append(row)
S = pd.DataFrame(SUMMARY); display(S.drop(columns=[c for c in S.columns if c.endswith("_ci")]))

PAIRS = []
for slm in CFG.SLMS:
    full = D[(D.slm == slm) & (D.cond == "D")].sort_values("rec")
    for cond in "ABC":
        other = D[(D.slm == slm) & (D.cond == cond)].sort_values("rec")
        for k in ["any_problem", "any_wrong", "any_overreach"]:
            a, b = other[k].to_numpy(), full[k].to_numpy()
            n10, n01 = int((a & ~b).sum()), int((~a & b).sum())
            p = binomtest(n10, n10 + n01, .5).pvalue if n10 + n01 else 1.0
            diff, ci = boot_diff(a, b)
            PAIRS.append(dict(slm=slm, endpoint=k, vs=f"{CONDS[cond]} - full", only_other=n10, only_full=n01,
                              mcnemar_p=float(p), diff=diff, ci95=ci))
        a, b = other.cert_right_features.to_numpy(), full.cert_right_features.to_numpy()
        diff, ci = boot_diff(a, b)
        PAIRS.append(dict(slm=slm, endpoint="cert_right_features", vs=f"{CONDS[cond]} - full", diff=diff, ci95=ci))
P = pd.DataFrame(PAIRS); display(P)

In [ ]:
# LaTeX rows for Table IV of the paper (one block per SLM)
L = [r"% condition & alerts with a certified wrong fact & over-reach & fabricated feature & any & certified facts/alert & empty or fallback & latency (ms)"]
for slm in CFG.SLMS:
    L.append(r"\multicolumn{8}{@{}l}{\emph{" + slm + r"}}\\")
    for r in [x for x in SUMMARY if x["slm"] == slm]:
        lat = r["gen_ms"] + r["check_ms"] + r["plan_ms"]
        ef = r["empty_rate"] if r["cond"] == "B" else r["fallback_rate"]
        L.append(f"{r['name']} & {100 * r['any_wrong_rate']:.1f} & {100 * r['any_overreach_rate']:.1f} & "
                 f"{100 * r['any_fabricated_rate']:.1f} & {100 * r['any_problem_rate']:.1f} & "
                 f"{r['cert_right_features']:.2f} & {100 * ef:.1f} & {lat:,.0f}" + r" \\")
HEAD = [r"\begin{table}[t]", r"\centering",
        r"\caption{Ablation on the 300 alerts of \Ttwo{} (flows never used in development). Share of alerts (\%) "
        r"whose delivered explanation contains a wrong fact certified by an independent parser, an over-reaching "
        r"sentence, or a feature absent from the schema; certified correct facts per alert; share of alerts with "
        r"nothing delivered (check) or with the planned statements delivered verbatim (audit); median latency per "
        r"alert on the T4.}", r"\label{tab:ablation}", r"\setlength{\tabcolsep}{2.2pt}", r"\footnotesize",
        r"\resizebox{\columnwidth}{!}{%", r"\begin{tabular}{@{}lccccccr@{}}", r"\toprule",
        r"& Wrong & Over- & Fabri- & Any & Facts & Empty/ & Latency\\",
        r"Condition & fact & reach & cated & & /alert & fallback & (ms)\\", r"\midrule"]
TAIL = [r"\bottomrule", r"\end{tabular}}", r"\end{table}"]
TEX = os.path.join(CFG.WORK, "ablation_table.tex"); open(TEX, "w").write("\n".join(HEAD + L[1:] + TAIL))
RES = dict(meta=dict(dryrun=DRYRUN, freeze_sha256=SHA, run_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                     alerts=len(ALERTS), plan_statements_per_alert=float(np.mean(kept)), plan_dropped=dropped,
                     plan_ms=PLAN_MS), summary=SUMMARY, paired=PAIRS, cost=COST)
json.dump(RES, open(os.path.join(CFG.WORK, "ablation_results.json"), "w"), indent=1, default=str)
print("\n".join(L)); print("\nsaved in", CFG.WORK)

## 8. How to read the results

* B and D cannot deliver a sentence that the verifier refutes; the column `v1_refuted` is therefore 0
  **by construction** and is not an endpoint. The endpoints come from instruments that do not call the
  compiler; a non-zero rate in B or D is a statement that slipped past the compiler.
* The certifying parser is strict: it measures a *lower bound* on wrong facts (it ignores sentences it
  cannot parse unambiguously) and the same lower bound on correct facts, identically in all conditions.
* The lexicons are indicators, not gold labels; section 9 gives the human check.

In [ ]:
if DRYRUN:
    print("Dry run: nothing here is reportable.")
else:
    for slm in CFG.SLMS:
        rs = {r["cond"]: r for r in SUMMARY if r["slm"] == slm}
        print(f"\n{slm}: alerts with any problem  A {rs['A']['any_problem_rate']:.3f}  B {rs['B']['any_problem_rate']:.3f}"
              f"  C {rs['C']['any_problem_rate']:.3f}  D {rs['D']['any_problem_rate']:.3f}")
        print(f"   certified correct features/alert  A {rs['A']['cert_right_features']}  B {rs['B']['cert_right_features']}"
              f"  C {rs['C']['cert_right_features']}  D {rs['D']['cert_right_features']}")
        print(f"   B empty {rs['B']['empty_rate']:.3f} | D fallback {rs['D']['fallback_rate']:.3f}, "
              f"flagged sentences/alert {rs['D']['flagged_per_alert']}")
        for p in PAIRS:
            if p["slm"] == slm and p["endpoint"] == "any_problem":
                print(f"   {p['vs']:28s} diff {p['diff']:+.3f} {p['ci95']}  McNemar p={p['mcnemar_p']:.1e}")

## 9. Blind annotation sheet (optional, ≈ 1 h for one annotator)

25 alerts per SLM × 4 conditions = 200 texts, shuffled, condition hidden. For each row, read the
record and mark `wrong_fact` (a stated value/band/class contradicted by the record), `overreach` (cause,
intent or certainty not established by one flow) and `useful` (1 = nothing usable, 2 = partly, 3 = useful).
Save the filled file as `annotation_filled.csv` in the same folder and re-run this cell.

In [ ]:
ANN = os.path.join(CFG.WORK, "annotation_sheet.csv"); KEY = os.path.join(CFG.WORK, "annotation_key.csv")
if not os.path.exists(ANN):
    g = np.random.default_rng(CFG.SEED2 + 1); pick = sorted(g.choice(ALERTS, CFG.N_ANNOT, replace=False).tolist())
    sh = D[D.rec.isin(pick)][["slm", "cond", "rec", "yhat", "text"]].sample(frac=1, random_state=7).reset_index(drop=True)
    sh["item"] = [f"I{i:03d}" for i in range(len(sh))]
    sh[["item", "slm", "cond", "rec"]].to_csv(KEY, index=False)
    sh["record"] = sh.rec.map(lambda j: record_text(j).replace("\n", " | "))
    for c in ["wrong_fact", "overreach", "useful"]: sh[c] = ""
    sh[["item", "yhat", "record", "text", "wrong_fact", "overreach", "useful"]].to_csv(ANN, index=False)
    tag("ANNOT", f"blind sheet written: {ANN} ({len(sh)} rows); key kept separately")
FILLED = os.path.join(CFG.WORK, "annotation_filled.csv")
if os.path.exists(FILLED):
    a = pd.read_csv(FILLED).merge(pd.read_csv(KEY), on="item")
    for c in ["wrong_fact", "overreach", "useful"]: a[c] = pd.to_numeric(a[c], errors="coerce")
    display(a.groupby(["slm", "cond"])[["wrong_fact", "overreach", "useful"]].mean().round(3))
    m = a.merge(D[["slm", "cond", "rec", "any_wrong", "any_overreach"]], on=["slm", "cond", "rec"])
    print("agreement instrument vs annotator: wrong", round(float((m.any_wrong == (m.wrong_fact > 0)).mean()), 3),
          "| over-reach", round(float((m.any_overreach == (m.overreach > 0)).mean()), 3))

## 10. Audit injection test with V1 (CPU, < 1 min)

This section tests the audit with the
final verifier V1 on the 600 plan-constrained texts already generated (300 alerts × 2 SLMs; no new
generation). Into each text we insert, separately, one sentence of each kind at a random position:
a wrong value of a planned feature (1.5x + 3), the opposite magnitude of a planned feature, and a
fabricated feature (*payload entropy*). The unit is the sentence. We report, per kind, injections
removed by the audit (TP) and kept (FN); and, for the SLM's own sentences, how many the audit removes,
split by whether the independent instruments flag them. The design is hashed before it runs.

In [ ]:
INJ = dict(value="The {f} is {v}.", magnitude="The {f} is {b} compared with other flows.",
           fabricated="The payload entropy of this flow is {v}.", seed=CFG.SEED2 + 7, unit="sentence")
INJ_SHA = hashlib.sha256(json.dumps(INJ, sort_keys=True).encode()).hexdigest()
INJ_FREEZE = os.path.join(CFG.WORK, "injection_freeze.json")
if os.path.exists(INJ_FREEZE):
    assert json.load(open(INJ_FREEZE))["sha256"] == INJ_SHA, "injection design changed after freezing: STOP"
else:
    json.dump(dict(sha256=INJ_SHA, frozen_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(), design=INJ),
              open(INJ_FREEZE, "w"), indent=1)
INV = {"low": "high", "high": "low"}
def removed(s, j, cstar): return is_meta(s) or not audit_match(s, j, cstar)
g = np.random.default_rng(INJ["seed"]); R = []; ORIG = []
for slm in CFG.SLMS:
    PL = cached(slm, "plan")
    for j in ALERTS:
        cstar = PLANS[j][0]; sents = split_sentences(PL[j])
        for s in sents:
            m = measure([s], j)
            ORIG.append(dict(slm=slm, rec=j, removed=removed(s, j, cstar),
                             instr_problem=bool(m["cert_wrong"] or m["overreach"] or m["fabricated"])))
        feats_planned = [p["feature"] for p in cstar if p["typ"] == "VALUE" and p["feature"]]
        if not feats_planned: continue
        f = feats_planned[0]; x = real_values(j)[f]; b = _band(f, x)
        inj = {"value": INJ["value"].format(f=disp(f), v=fmt(1.5 * x + 3)),
               "magnitude": INJ["magnitude"].format(f=disp(f), b=INV[b]),
               "fabricated": INJ["fabricated"].format(v=fmt(x))}
        assert abs(1.5 * x + 3 - x) > TOL * max(abs(x), 1e-6)
        for kind, s_inj in inj.items():
            pos = int(g.integers(0, len(sents) + 1)); txt = sents[:pos] + [s_inj.rstrip(".")] + sents[pos:]
            again = split_sentences(" ".join(t + "." for t in txt))
            hit = [s for s in again if s.strip(" .") == s_inj.rstrip(".")]
            R.append(dict(slm=slm, rec=j, kind=kind, split_ok=len(hit) == 1,
                          removed=removed(hit[0], j, cstar) if hit else None))
R = pd.DataFrame(R); O = pd.DataFrame(ORIG)
print("injections whose sentence survived re-splitting:", int(R.split_ok.sum()), "/", len(R))
T = R[R.split_ok].groupby(["kind"]).agg(n=("removed", "size"), TP_removed=("removed", "sum"))
T["FN_kept"] = T.n - T.TP_removed; T["recall"] = (T.TP_removed / T.n).round(3); display(T)
display(R[R.split_ok].groupby(["slm", "kind"]).removed.agg(["size", "sum"]))
OS = O.groupby(["slm", "instr_problem"]).removed.agg(n="size", removed="sum"); OS["share"] = (OS.removed / OS.n).round(3)
print("SLM's own sentences removed by the audit, split by the independent instruments:"); display(OS)
INJ_RES = dict(sha256=INJ_SHA, injections=T.reset_index().to_dict("records"),
               by_slm=R[R.split_ok].groupby(["slm", "kind"]).removed.agg(["size", "sum"]).reset_index().to_dict("records"),
               own_sentences=OS.reset_index().to_dict("records"), n_texts=len(ALERTS) * len(CFG.SLMS),
               texts_without_planned_value=int(len(ALERTS) * len(CFG.SLMS) - len(R) / 3))
json.dump(INJ_RES, open(os.path.join(CFG.WORK, "injection_v1_results.json"), "w"), indent=1, default=str)
print("saved injection_v1_results.json")

## 11. Data partition and environment (for the reproducibility table)

In [ ]:
import glob, platform
part = dict(test_flows=int(Xraw.shape[0]), tdev_ttest_flows=len(flows), ttest2_flows=len(ALERTS),
            overlap_tdev_ttest2=len(set(flows) & set(ALERTS)))
def _idx(path):
    d = pd.read_csv(path); low = {c.lower(): c for c in d.columns}
    c = next((low[k] for k in ["idx", "flow_idx", "flow_id", "index", "j", "record", "row"] if k in low), None)
    return set(d[c].astype(int).tolist()) if c else set()
for name, pat in [("tinyllama_rationales", "/**/soc_eval_*" + CFG.DATASET + "*.csv"), ("qwen_rationales", "/**/*[Qq][Ww][Ee][Nn]*.csv")]:
    hits = sorted(h for h in glob.glob(CFG.DRIVE_ROOT + pat, recursive=True)
                  if not os.path.abspath(h).startswith(os.path.abspath(CFG.WORK)) and "/ttest2/" not in h)
    for h in hits[:3]:
        try:
            s = _idx(h)
            part[f"{name}:{os.path.basename(h)}"] = dict(flows=len(s), in_tdev_ttest=len(s & set(flows)), in_ttest2=len(s & set(ALERTS)))
        except Exception as e:
            part[f"{name}:{os.path.basename(h)}"] = f"unreadable: {e}"
cpu = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor())
env = dict(cpu=cpu, cores=os.cpu_count(), python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__)
try:
    import torch, transformers
    env.update(torch=torch.__version__, transformers=transformers.__version__,
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
except Exception: pass
env["latency_protocol"] = (f"median of {CFG.N_LATENCY} single-alert generate() calls per model and prompt, after the batched "
                           "generation of all 300 alerts (warm model), torch.cuda.synchronize() around each call; "
                           "plan and audit timed per alert with time.perf_counter on the CPU")
print(json.dumps(part, indent=1)); print(json.dumps(env, indent=1))
json.dump(dict(partition=part, environment=env), open(os.path.join(CFG.WORK, "partition_env.json"), "w"), indent=1)

## 12. Tercile sensitivity (CPU, ~2 min)

All published results use magnitude bands (terciles) computed without labels on the 1,800 test flows.
Here the terciles are recomputed on references that exclude every evaluated flow: (a) the 1,500 test
flows outside T_test2 / the ablation, and (b) the detector's training split, if `X_train_sc.npy` is
found. Under each reference we recompute, on the same texts: the reference verdicts and V1's verdicts on
T_test2 (MiniCheck's outputs are fixed), and the ablation's certified wrong facts. Nothing is regenerated;
the published numbers stay the primary result, this is a sensitivity analysis.

In [ ]:
import copy, glob
FQ_TEST = copy.deepcopy(feat_q)
def terciles(X):
    return {f: {"q33": float(np.percentile(X[:, i], 33)), "q66": float(np.percentile(X[:, i], 66)),
                "med": float(np.median(X[:, i]))} for i, f in enumerate(feats)}
REFS = {"test_outside_eval (1,500 flows)": terciles(Xraw[sorted(set(range(Xraw.shape[0])) - set(ALERTS))])}
if not DRYRUN:
    import joblib
    for nm in ["X_train_sc.npy", "X_train.npy"]:
        p = os.path.join(CFG.TINY_DIR, nm)
        if os.path.exists(p):
            Xt = np.load(p); Xt = joblib.load(os.path.join(CFG.TINY_DIR, "scaler.joblib")).inverse_transform(Xt) if "sc" in nm else Xt
            REFS[f"train split ({Xt.shape[0]} flows)"] = terciles(Xt); break
    else:
        tag("TERCILE", "no training-split file found in TINY_DIR: only the 1,500-flow reference is used")
EXEC_G = {"VAL", "CMP", "QNT", "ABN", "CLS"}
T2 = pd.read_csv(os.path.join(CFG.TTEST2_DIR, "ttest2_claims.csv")); T2 = T2[T2.set == "ttest2"].reset_index(drop=True)
BAND_T = {"N05", "N06", "N08"}; QUANT = {"N09": "few", "N10": "many", "N11": "few"}; BEHAV_T = {"N16": "denial of service", "N17": "brute-force"}
def gold_under(r):
    rv = real_values(int(r.rec)); f = r.feat
    if r.tid in BAND_T:
        claimed = re.findall(r"(low|medium|high)", r.text.lower())[-1]
        return "SUPPORTED" if _band(f, rv[f]) == claimed else "CONTRADICTED"
    if r.tid == "N07": return "SUPPORTED" if _band(f, rv[f]) == "medium" else "CONTRADICTED"
    if r.tid in QUANT:
        b = _band(f, rv[f])
        if b == "medium": return "UNRESOLVED"
        return "SUPPORTED" if (b == "low") == (QUANT[r.tid] == "few") else "CONTRADICTED"
    if r.tid in BEHAV_T: return bcon_gold(BEHAV_T[r.tid], int(r.rec))
    return r.gold
def h2_metrics(gold, v1):
    neu = (T2.stratum == "neutral").values; comp = neu & T2.true_group.isin(EXEC_G).values; pr = (T2.stratum == "probe").values
    out = {}
    for name, pred in [("V1", v1), ("RoBERTa-L", T2["MiniCheck-roberta-large"].values)]:
        s = pred == "SUPPORTED"; pos = gold == "SUPPORTED"
        tp, fp = int((s & pos & comp).sum()), int((s & ~pos & comp).sum())
        out[name] = dict(true_computable=int((pos & comp).sum()), approved_true=tp, wrong_approvals_computable=fp,
                         precision_computable=round(tp / max(tp + fp, 1), 3), coverage=round(tp / max(int((pos & comp).sum()), 1), 3),
                         wrong_approvals_neutral=int((s & ~pos & neu).sum()), wrong_approvals_probes=int((s & ~pos & pr).sum()))
        acc_c = ((s == pos) & comp).sum() / comp.sum(); out[name]["approval_accuracy_computable"] = round(float(acc_c), 4)
    return out
ABL = pd.read_csv(os.path.join(CFG.WORK, "ablation_alerts.csv"), keep_default_na=False)
def abl_metrics():
    rows = []
    for _, r in ABL.iterrows():
        sents = [s.strip().rstrip(".") for s in re.split(r"(?<=\.)\s+", str(r.text)) if s.strip()]
        m = measure(sents, int(r.rec))
        rows.append(dict(slm=r.slm, cond=r.cond, wrong=m["cert_wrong"] > 0,
                         problem=(m["cert_wrong"] > 0) or bool(r.any_overreach) or bool(r.any_fabricated)))
    A = pd.DataFrame(rows)
    return A.groupby(["slm", "cond"])[["wrong", "problem"]].mean().round(3).reset_index().to_dict("records")
SENS = {}
for label, FQ in [("published (test terciles)", FQ_TEST)] + list(REFS.items()):
    feat_q = FQ
    ch = float(np.mean([_band(f, real_values(j)[f]) != ("low" if real_values(j)[f] < FQ_TEST[f]["q33"] else ("high" if real_values(j)[f] > FQ_TEST[f]["q66"] else "medium"))
                        for j in ALERTS for f in feats]))
    gold = T2.apply(gold_under, axis=1).values
    v1 = np.array([V1(t, int(j))[0] for t, j in zip(T2.text, T2.rec)])
    SENS[label] = dict(band_changes_on_eval_values=round(ch, 4), gold_changes=int((gold != T2.gold.values).sum()),
                       h2=h2_metrics(gold, v1), ablation=abl_metrics())
    print(f"\n=== {label}: {100 * ch:.1f}% of band assignments change on the evaluated flows; "
          f"{SENS[label]['gold_changes']} of {len(T2)} reference verdicts change")
    for k, v in SENS[label]["h2"].items(): print("  ", k, v)
    A = pd.DataFrame(SENS[label]["ablation"]); print(A.pivot(index="slm", columns="cond", values="problem").to_string())
feat_q = FQ_TEST
pub = SENS["published (test terciles)"]["h2"]["V1"]
if not DRYRUN: assert SENS["published (test terciles)"]["gold_changes"] == 0, "reference verdicts not reproduced: STOP"
if not DRYRUN: assert pub["approved_true"] == 1844 and pub["wrong_approvals_neutral"] == 126, "published numbers not reproduced: STOP"
tag("TERCILE", "published numbers reproduced with the test terciles; feat_q restored")
json.dump(SENS, open(os.path.join(CFG.WORK, "tercile_sensitivity.json"), "w"), indent=1, default=str)
print("saved tercile_sensitivity.json")

## 13. Natural-sentence evaluation (Sec. V-E) — phase 1: sample, seal, blind sheet

Templates were written by the authors. Here the verifier is judged on sentences **written by the SLMs
themselves** (the free explanations of the 300 T_test2 alerts, generated after the compiler was frozen),
with **human gold labels**, the protocol of AggreFact / LLM-AggreFact used to evaluate fact-checkers.

* Population: every non-meta sentence of the free explanations of both SLMs.
* Stratified sample (seed fixed): 75 sentences per SLM × {approved by V1, not approved}; population weights
  are kept so that estimates are reweighted to the population (Horvitz–Thompson).
* V1 and MiniCheck RoBERTa-L verdicts are computed **now** and sealed with the sample (SHA-256) **before**
  any human label exists; they are written to a key file that annotators never see.
* Output: `natural_sheet_blind.xlsx` (to annotate), `natural_key.csv` (hidden), `natural_freeze.json`.

Annotators judge the **whole sentence** against the record: SUPPORTED (everything it asserts is
established by the record or the decision), CONTRADICTED (at least one assertion is false), UNVERIFIABLE
(an assertion one flow cannot establish, no false one), NO CLAIM (asserts nothing about this flow).

In [ ]:
NAT = dict(per_stratum=75, seed=CFG.SEED2 + 13, labels=["SUPPORTED", "CONTRADICTED", "UNVERIFIABLE", "NO CLAIM"],
           approval="V1 verdict == SUPPORTED", baseline="MiniCheck-roberta-large, full premise, default rule",
           premise="The <f> is <round(v,3)> (<band> range). x25 + The classifier predicts <yhat>.")
NAT_DIR = os.path.join(CFG.WORK, "natural"); os.makedirs(NAT_DIR, exist_ok=True)
NAT_FREEZE = os.path.join(NAT_DIR, "natural_freeze.json"); NAT_KEY = os.path.join(NAT_DIR, "natural_key.csv")
NAT_SHEET = os.path.join(NAT_DIR, "natural_sheet_blind.xlsx")

def premise_full(j):
    rv = real_values(j)
    return " ".join(f"The {f.replace('_', ' ')} is {round(rv[f], 3)} ({_band(f, rv[f])} range)." for f in feats) \
           + f" The classifier predicts {y_str[j]}."

if os.path.exists(NAT_FREEZE):
    tag("NATURAL", f"sample already sealed ({json.load(open(NAT_FREEZE))['frozen_utc']}): phase 1 is not re-run")
    POP_N = json.load(open(NAT_FREEZE))["population"]
else:
    rows = []
    for slm in CFG.SLMS:
        FREE = cached(slm, "free")
        for j in ALERTS:
            prev = ""
            for s in split_sentences(FREE[j]):
                if is_meta(s): prev = s; continue
                v, br, cat = V1(s, j, y_str[j])
                rows.append(dict(slm=slm, rec=j, yhat=y_str[j], sentence=s, previous=prev, V1_verdict=v, V1_branch=br,
                                 V1_type=cat, V1_approve=(v == "SUPPORTED")))
                prev = s
    POPU = pd.DataFrame(rows).drop_duplicates(["slm", "rec", "sentence"]).reset_index(drop=True)
    POP_N = POPU.groupby(["slm", "V1_approve"]).size().rename("N").reset_index()
    print("population of sentences:"); print(POP_N)
    g = np.random.default_rng(NAT["seed"]); parts = []
    for (slm, ap), grp in POPU.groupby(["slm", "V1_approve"]):
        k = min(NAT["per_stratum"], len(grp))
        parts.append(grp.iloc[sorted(g.choice(len(grp), k, replace=False))].assign(N_h=len(grp), n_h=k))
    S = pd.concat(parts).reset_index(drop=True); S["weight"] = S.N_h / S.n_h
    # MiniCheck verdicts, computed before any human label
    if DRYRUN:
        S["MC_approve"] = np.random.default_rng(5).random(len(S)) < 0.5
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"], check=False)
        import nltk; [nltk.download(p, quiet=True) for p in ("punkt", "punkt_tab")]
        import minicheck.inference as _mci; _mci.tqdm = lambda it, *a, **k: it
        from minicheck.minicheck import MiniCheck
        mc = MiniCheck(model_name="roberta-large", cache_dir="./ckpts")
        lab, _, _, _ = mc.score(docs=[premise_full(int(j)) for j in S.rec], claims=S.sentence.tolist())
        S["MC_approve"] = [int(l) == 1 for l in lab]
        del mc; gc.collect()
    S = S.sample(frac=1, random_state=NAT["seed"]).reset_index(drop=True)
    S["item"] = [f"N{i:03d}" for i in range(len(S))]
    S.to_csv(NAT_KEY, index=False)
    payload = json.dumps(dict(design=NAT, items=S[["item", "slm", "rec", "sentence", "V1_verdict", "MC_approve"]].astype(str).values.tolist()), sort_keys=True)
    json.dump(dict(sha256=hashlib.sha256(payload.encode()).hexdigest(), frozen_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                   design=NAT, population=POP_N.to_dict("records"), n_items=len(S)), open(NAT_FREEZE, "w"), indent=1)
    tag("NATURAL", f"{len(S)} sentences sampled and sealed with V1 and MiniCheck verdicts -> {NAT_FREEZE}")
    # blind Excel sheet
    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
    from openpyxl.worksheet.datavalidation import DataValidation
    F = "Arial"; thin = Side(style="thin", color="C9D1CE"); B = Border(top=thin, bottom=thin, left=thin, right=thin)
    wb = Workbook(); gd = wb.active; gd.title = "Read me"; gd.column_dimensions["A"].width = 26; gd.column_dimensions["B"].width = 100
    # The annotators received this guide in French; the text below is its English translation (same content).
    guide = [("Annotation of SLM-written sentences", None),
     ("What to do", "For each row of the 'Annotation' sheet, read the SENTENCE (column E) and judge it AS A WHOLE against the flow record (column C) and the decision (column B). Pick a label in column F (drop-down list). Column D (previous sentence) is context only, to resolve 'this', 'it', etc.; it is not judged."),
     ("SUPPORTED", "Everything the sentence asserts is established by the record or the decision: exact values (to the written rounding or within ~2%), correct bands (high/low/medium), class = decision, absence of a zero value. Paraphrased feature names are accepted ('total size' = tot size)."),
     ("CONTRADICTED", "At least one assertion is false given the record: different value, opposite band, other class, invented feature or number (port, IP, invented unit...). If the sentence contains both an error and an interpretation, choose CONTRADICTED."),
     ("UNVERIFIABLE", "No error, but at least one assertion that a single flow cannot establish: intent, attacker, goal, consequence, cause of the behaviour, certainty ('clearly'), comparison with 'typical traffic' without a value. A true sentence followed by an interpretation ('..., indicating that the attacker...') is UNVERIFIABLE."),
     ("NO CLAIM", "The sentence asserts nothing about this flow (general definition of an attack, transition, truncated sentence without content)."),
     ("Edge cases", "'because X is high' used only to say why the detector decided: judge only 'X is high'. 'compatible with <behaviour>': SUPPORTED if the rest is exact (compatibility is not an intent). When in doubt between SUPPORTED and UNVERIFIABLE, choose UNVERIFIABLE."),
     ("Blind", "Neither the model nor the verdicts of the verifier or of MiniCheck appear in this file. Do not discuss with the other annotator before finishing."),
     ("When finished", "Save the file as natural_sheet_A1.xlsx (or A2 for the second annotator) in the natural/ folder, then re-run the notebook (section 14).")]
    for r, (a, b) in enumerate(guide, 1):
        gd.cell(r, 1, a).font = Font(name=F, bold=True, size=12 if r == 1 else 10)
        gd.cell(r, 1).alignment = Alignment(vertical="top", wrap_text=True)
        if b: c = gd.cell(r, 2, b); c.font = Font(name=F, size=10); c.alignment = Alignment(wrap_text=True, vertical="top")
    gd.cell(len(guide) + 2, 1, "Progress").font = Font(name=F, bold=True, size=10)
    gd.cell(len(guide) + 2, 2, f'=COUNTA(Annotation!F2:F{len(S)+1})&" / {len(S)} sentences labelled"').font = Font(name=F, size=10, bold=True)
    ws = wb.create_sheet("Annotation")
    heads = [("Item", 7), ("Decision", 10), ("Flow record (value, band)", 30), ("Previous sentence (context)", 34), ("SENTENCE TO JUDGE", 60), ("Label", 16), ("Comment", 28)]
    for k, (h, w) in enumerate(heads, 1):
        c = ws.cell(1, k, h); c.font = Font(name=F, bold=True, color="FFFFFF", size=10); c.fill = PatternFill("solid", fgColor="1F4E79")
        c.alignment = Alignment(wrap_text=True, horizontal="center", vertical="center"); ws.column_dimensions[c.column_letter].width = w
    for i, r in S.iterrows():
        j = int(r.rec); rec = "\n".join(f"{f.replace('_', ' ')}: {fmt(real_values(j)[f])} ({_band(f, real_values(j)[f])})" for f in feats)
        for k, v in enumerate([r["item"], r.yhat, rec, r.previous, r.sentence, None, None], 1):
            c = ws.cell(i + 2, k, v); c.font = Font(name=F, size=8 if k == 3 else 10, bold=(k == 5)); c.border = B
            c.alignment = Alignment(wrap_text=True, vertical="top")
            if k == 6: c.fill = PatternFill("solid", fgColor="FFF6C9")
        ws.row_dimensions[i + 2].height = 260
    dv = DataValidation(type="list", formula1='"SUPPORTED,CONTRADICTED,UNVERIFIABLE,NO CLAIM"', allow_blank=True)
    ws.add_data_validation(dv); dv.add(f"F2:F{len(S)+1}"); ws.freeze_panes = "E2"
    wb.save(NAT_SHEET); tag("NATURAL", f"blind annotation sheet -> {NAT_SHEET}")

## 14. Natural-sentence evaluation (Sec. V-E) — phase 2: results (after annotation)

Put the filled sheets in the `natural/` folder as `natural_sheet_A1.xlsx` and `natural_sheet_A2.xlsx`, then run
again. For each annotator: reweighted approval precision (share of approved sentences judged SUPPORTED) and
coverage (share of SUPPORTED sentences approved), for V1 and MiniCheck RoBERTa-L, with an alert-cluster
bootstrap; agreement between annotators; what V1's non-SUPPORTED approvals are (false vs unverifiable).

In [ ]:
from sklearn.metrics import cohen_kappa_score
def read_sheet(p):
    from openpyxl import load_workbook
    ws = load_workbook(p, data_only=True)["Annotation"]
    return pd.DataFrame([dict(item=r[0], label=(str(r[5]).strip().upper() if r[5] else None), comment=r[6])
                         for r in ws.iter_rows(min_row=2, values_only=True) if r[0]])
KEYN = pd.read_csv(NAT_KEY); ANN_N = {}
_fz = json.load(open(NAT_FREEZE))
_pl = json.dumps(dict(design=_fz["design"], items=KEYN[["item", "slm", "rec", "sentence", "V1_verdict", "MC_approve"]].astype(str).values.tolist()), sort_keys=True)
assert hashlib.sha256(_pl.encode()).hexdigest() == _fz["sha256"], "natural_key.csv differs from the sealed sample: STOP"
tag("NATURAL", f"key matches the sealed sample (sha {_fz['sha256'][:8]}...)")
for a in ("A1", "A2"):
    p = os.path.join(NAT_DIR, f"natural_sheet_{a}.xlsx")
    if os.path.exists(p):
        d = read_sheet(p); ANN_N[a] = d; tag("NATURAL", f"{a}: {d.label.notna().sum()} / {len(d)} labelled")
if not ANN_N:
    print("No filled sheet yet: annotate natural_sheet_blind.xlsx, save as natural_sheet_A1.xlsx / _A2.xlsx, re-run.")
else:
    def metrics(D, appr, w="weight"):
        sup = D.label == "SUPPORTED"; a = D[appr].astype(bool)
        prec = (D[w] * (a & sup)).sum() / max((D[w] * a).sum(), 1e-9)
        cov = (D[w] * (a & sup)).sum() / max((D[w] * sup).sum(), 1e-9)
        return prec, cov
    def boot(D, appr, B=CFG.B_BOOT, seed=CFG.SEED2):
        g = np.random.default_rng(seed); est = []
        groups = {k: v for k, v in D.groupby(["slm", "rec"])}; keys = list(groups)
        by_slm = {s: [k for k in keys if k[0] == s] for s in D.slm.unique()}
        for _ in range(B):
            pick = [groups[ks[i]] for s, ks in by_slm.items() for i in g.integers(0, len(ks), len(ks))]
            est.append(metrics(pd.concat(pick), appr))
        e = np.array(est); return np.percentile(e[:, 0], [2.5, 97.5]).round(3).tolist(), np.percentile(e[:, 1], [2.5, 97.5]).round(3).tolist()
    NRES = {}
    for a, d in ANN_N.items():
        D = KEYN.merge(d, on="item"); D = D[D.label.notna() & (D.label != "NO CLAIM")]
        NRES[a] = {"n_labelled": int(len(D)),
                   "label_shares_weighted": (D.groupby("label").weight.sum() / D.weight.sum()).round(3).to_dict()}
        for name, col in [("V1", "V1_approve"), ("MiniCheck-RoBERTa-L", "MC_approve")]:
            p, c = metrics(D, col); ci_p, ci_c = boot(D, col, B=2000)
            NRES[a][name] = dict(precision=round(p, 3), precision_ci=ci_p, coverage=round(c, 3), coverage_ci=ci_c)
            for slm in D.slm.unique():
                ps, cs = metrics(D[D.slm == slm], col); NRES[a][name][slm] = dict(precision=round(ps, 3), coverage=round(cs, 3))
        g = np.random.default_rng(CFG.SEED2); groups = [v for _, v in D.groupby(["slm", "rec"])]; dif = []
        for _ in range(2000):
            R = pd.concat([groups[i] for i in g.integers(0, len(groups), len(groups))])
            dif.append(metrics(R, "V1_approve")[0] - metrics(R, "MC_approve")[0])
        d0 = NRES[a]["V1"]["precision"] - NRES[a]["MiniCheck-RoBERTa-L"]["precision"]
        NRES[a]["precision_diff_V1_minus_MC"] = dict(diff=round(d0, 3), ci=np.percentile(dif, [2.5, 97.5]).round(3).tolist())
        app = D[D.V1_approve]
        NRES[a]["V1_approved_not_supported"] = (app[app.label != "SUPPORTED"].groupby("label").weight.sum() / app.weight.sum()).round(3).to_dict()
        print(f"\n=== annotator {a} ===\n" + json.dumps(NRES[a], indent=1))
    if len(ANN_N) == 2:
        M2 = ANN_N["A1"].merge(ANN_N["A2"], on="item", suffixes=("_1", "_2")).dropna(subset=["label_1", "label_2"])
        k4 = cohen_kappa_score(M2.label_1, M2.label_2)
        k2 = cohen_kappa_score(M2.label_1 == "SUPPORTED", M2.label_2 == "SUPPORTED")
        NRES["agreement"] = dict(n=len(M2), raw=round(float((M2.label_1 == M2.label_2).mean()), 3), kappa_4class=round(k4, 3), kappa_supported_vs_not=round(k2, 3))
        print("\nagreement:", NRES["agreement"])
    json.dump(NRES, open(os.path.join(NAT_DIR, "natural_results.json"), "w"), indent=1, default=float)
    tag("NATURAL", "saved natural/natural_results.json")